# 0. Импорты, пути, конфиг

In [1]:
# 0. ИМПОРТЫ И ПОДКЛЮЧЕНИЕ src/
# ============================================================================
# Единственное место, где импортируются модули из src/.
# Все остальные ячейки используют то, что здесь загружено.
# ============================================================================

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd


# ---------- Подключаем src/ и src/detectors/ ----------
def find_src(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'src').is_dir() and (p / '01_data_analysis').is_dir():
            return p / 'src'
    raise FileNotFoundError('src/ не найден выше текущей директории')


SRC_PATH = find_src()
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))
if str(SRC_PATH / 'detectors') not in sys.path:
    sys.path.append(str(SRC_PATH / 'detectors'))


# ---------- Infrastructure ----------
from paths import (
    get_base_path, get_dashboard_path, get_datasets_path,
    get_config_path, ensure_dashboard_subdirs, ensure_datasets_subdirs,
)
from loaders import load_all

# ---------- Primitives ----------
from temporal import compute_windows, trim_cold_start
from enrich import (
    add_timestamps, add_valid_play, add_first_valid_seen,
    add_my_music, add_sessions, add_track_source,
)
# ---------- Aggregators ----------
from aggregators.music import build_monthly_sound, SOUND_FEATURES
from aggregators.behavior import (
    is_obsession_event, build_strong_obsessions,
)
from aggregators.temporal import (
    AGG_WINDOWS, build_time_series, build_time_series_meta,
)

# ---------- Clustering ----------
from clustering.kmeans import (
    find_optimal_k, fit_clustering, fit_pca, mark_characteristic,
)
from clustering.sound_core import (
    prepare_core, apply_to_window, CLUSTER_FEATURES,
)

# ---------- Detectors ----------
from detectors.pelt_sound import detect as pelt_sound_detect
from detectors.pelt_behavior import detect as pelt_behavior_detect
from detectors.cosine_identity import detect as cosine_detect
from detectors.storm import detect as storm_detect

# ---------- Changepoints (оркестратор) ----------
from detectors.changepoints import build as build_changepoints

# ---------- Variability ----------
from detectors.variability import run_variability_pipeline
from variability_export import export_variability

# ---------- Portrait ----------
from portrait import (
    build as build_portrait,
    build_axes,
)

from indices import build_kpi_summary

from mask import mask_all

print(f"✅ src/ подключён")

# Динамический статус загруженных модулей из src/
_src_str = str(SRC_PATH)
_src_mods = sorted([
    name for name, mod in sys.modules.items()
    if getattr(mod, '__file__', None)
    and _src_str in str(mod.__file__)
    and '__pycache__' not in str(mod.__file__)
])
print(f"   Модулей из src/: {len(_src_mods)}")
for _m in _src_mods:
    print(f"      {_m}")

✅ src/ подключён
   Модулей из src/: 24
      aggregators.behavior
      aggregators.music
      aggregators.temporal
      clustering
      clustering.kmeans
      clustering.sound_core
      detectors
      detectors.changepoints
      detectors.cosine_identity
      detectors.pelt_behavior
      detectors.pelt_sound
      detectors.storm
      detectors.variability
      detectors.variability_toolkit
      enrich
      helpers
      indices
      loaders
      mask
      paths
      pelt_common
      portrait
      temporal
      variability_export


# 1. Пути, папки, загрузка, окна


In [2]:
# 1. ЗАГРУЗКА ДАННЫХ И ENRICH
# ============================================================================
# 1. Пути и конфиги.
# 2. Загрузка сырых данных.
# 3. Enrich: добавление колонок (valid_play, period, my_music, ...).
# 4. Окна.
#
# После этой ячейки df полностью обогащён и готов для любых селекторов.
# ==============================================================================

# ---------- Пути ----------
BASE_PATH = get_base_path()
DASHBOARD_DATA_PATH = get_dashboard_path()
DATASETS_PATH = get_datasets_path()
CONFIG_PATH = get_config_path()

PATHS = ensure_dashboard_subdirs()
ensure_datasets_subdirs()
SOUND_PATH      = PATHS['sound']
TIME_PATH       = PATHS['time']
BEHAVIOR_PATH   = PATHS['behavior']
COLLECTION_PATH = PATHS['collection']

# ---------- Конфиги ----------
with open(CONFIG_PATH / 'indices_display.json', 'r', encoding='utf-8') as f:
    INDICES_DISPLAY = json.load(f)

# ---------- Загрузка данных ----------
data = load_all(BASE_PATH)
df = data['df']
tracks_exportify   = data['tracks_exportify']
playlist_links     = data['playlist_links']
library_tracks     = data['library_tracks']
owner_ids          = data['owner_ids']
playlists_metadata = data['playlists_metadata']

# ---------- Enrich ----------
df = add_timestamps(df)
df = add_valid_play(df)
df = add_first_valid_seen(df)

df, my_music_uris = add_my_music(df, library_tracks, playlist_links, owner_ids)

df = add_sessions(df)
df = add_track_source(df, library_tracks, playlist_links, owner_ids)

# ---------- Окна ----------
meta = compute_windows(df, playlist_links)
max_data_date = meta['max_data_date']
recent_start  = meta['recent_start']
last90_start  = meta['last90_start']
windows       = meta['windows']

# ---------- Статус ----------
print('✅ Ячейка 1 завершена')

# Динамический статус df
print(f'\n📊 df: {len(df):,} строк, {len(df.columns)} колонок')
print(f'   Периоды:  {df["period"].min()} — {df["period"].max()} '
      f'({df["period"].nunique()} мес)')

_checks = ['ts_naive', 'period', 'valid_play', 'first_valid_seen',
           'my_music', 'source', 'session_id']
_present = [c for c in _checks if c in df.columns]
_missing = [c for c in _checks if c not in df.columns]
print(f'   Колонки:  {", ".join(_present)}')
if _missing:
    print(f'   ⚠️ Отсутствуют: {", ".join(_missing)}')

n_valid = int(df['valid_play'].sum())
print(f'   valid_play:    {n_valid:,} из {len(df):,} '
      f'({n_valid/len(df)*100:.1f}%)')

n_my = int(df['my_music'].sum())
print(f'   my_music:      {n_my:,} событий '
      f'({df.loc[df["my_music"], "spotify_track_uri"].nunique():,} треков)')

print(f'\n   Global:    {df["ts_naive"].min().date()} — '
      f'{df["ts_naive"].max().date()}')
print(f'   Recent 6Q: {recent_start.date()} — {max_data_date.date()}')
print(f'   Last 90d:  {last90_start.date()} — {max_data_date.date()}')

✅ Ячейка 1 завершена

📊 df: 79,752 строк, 58 колонок
   Периоды:  2020-05 — 2026-08 (72 мес)
   Колонки:  ts_naive, period, valid_play, first_valid_seen, my_music, source, session_id
   valid_play:    45,841 из 79,752 (57.5%)
   my_music:      43,437 событий (5,332 треков)

   Global:    2020-05-18 — 2026-08-03
   Recent 6Q: 2025-02-03 — 2026-08-03
   Last 90d:  2026-05-05 — 2026-08-03


# 2. SOUND: ЯДРО ВКУСА


In [19]:
# 2. SOUND: ЯДРО ВКУСА
# ============================================================================
from sklearn.preprocessing import StandardScaler

MIN_TRACKS_FOR_CLUSTERING = 30

MIN_SILHOUETTE = 0.25

# --- 1. Отбор ядра global ---
core_global = prepare_core(df, *windows['global'], my_music_uris)
if len(core_global) < MIN_TRACKS_FOR_CLUSTERING:
    raise SystemExit(f'Мало треков для кластеризации: {len(core_global)}')
print(f'Ядро global: {len(core_global)} треков')

# --- 2. Масштабирование + подбор k ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(core_global[CLUSTER_FEATURES].values)

best_k, best_sil, all_scores = find_optimal_k(X_scaled)
print(f'\nOptimal k={best_k}, silhouette={best_sil:.3f}')
for k, s in all_scores.items():
    print(f'   k={k}: {s:.3f}')

# --- 3. Обучение моделей ---
kmeans = fit_clustering(X_scaled, best_k)
pca = fit_pca(X_scaled)

# --- 4. Метаданные ---
pd.DataFrame([{
    'k': best_k,
    'silhouette': best_sil,
    'n_tracks': len(core_global),
    'threshold': MIN_SILHOUETTE,
    'passed': best_sil >= MIN_SILHOUETTE,
}]).to_parquet(SOUND_PATH / 'sound_clustering_meta.parquet', index=False)

# --- 5. Применение ко всем окнам ---
for label, (start, end) in windows.items():
    core = prepare_core(df, start, end, my_music_uris)
    if len(core) < MIN_TRACKS_FOR_CLUSTERING:
        print(f'⚠️ {label}: мало треков ({len(core)}), пропуск')
        continue

    core = apply_to_window(core, scaler, kmeans, pca)
    core = mark_characteristic(core, top_n=5)

    cols_to_save = [
        'track_name', 'artist_name', 'valence', 'energy', 'danceability',
        'acousticness', 'instrumentalness', 'popularity', 'genres_normalized',
        'cluster', 'play_count', 'pc1', 'pc2',
        'distance_to_center', 'is_characteristic',
    ]
    out = SOUND_PATH / f'sound_core_{label}.parquet'
    core[cols_to_save].to_parquet(out, index=False)
    print(f'💾 {out.name}: {len(core)} строк, характерных: {core["is_characteristic"].sum()}')

# --- 6. Контроль ---
for label in windows:
    path = SOUND_PATH / f'sound_core_{label}.parquet'
    if not path.exists():
        continue
    temp = pd.read_parquet(path)



Ядро global: 4374 треков

Optimal k=2, silhouette=0.264
   k=2: 0.264
   k=3: 0.230
   k=4: 0.222
   k=5: 0.231
   k=6: 0.230
💾 sound_core_global.parquet: 4374 строк, характерных: 10
💾 sound_core_recent_6q.parquet: 2486 строк, характерных: 10
💾 sound_core_last_90d.parquet: 484 строк, характерных: 10


# 3. TIME: ВРЕМЕННЫЕ РЯДЫ (три окна)


In [4]:
# 3. TIME: ВРЕМЕННЫЕ РЯДЫ (три окна)
# ============================================================================
# База: df без trim — полная история с холодным стартом и дыркой.
# Причина: дашборд должен показать всё, чтобы не было вопроса
# «где начало данных».
# ==============================================================================

# Граница плотной истории (после cold start)
df_t = trim_cold_start(df)
first_post_gap = df_t['period'].min()   # 2020-10

# ---------- Три окна ----------
for label, (start, end) in windows.items():
    ts = build_time_series(
        df, label, start, end, AGG_WINDOWS[label],
        first_post_gap_month=first_post_gap,
    )
    out = TIME_PATH / f'time_series_{label}.parquet'
    ts.to_parquet(out, index=False)
    n_gaps = int(ts['tracks'].isna().sum())
    print(f'💾 {out.name} ({len(ts)} строк, '
          f'пропусков: {n_gaps}, после gap: {ts["is_post_gap"].sum()})')

# ---------- Meta ----------
meta_ts = build_time_series_meta(df, first_post_gap)
meta_ts.to_parquet(TIME_PATH / 'time_series_meta.parquet', index=False)

print(f"\n📊 time_series_meta.parquet:")
print(f"   Первое событие:   {df['period'].min()}")
print(f"   После gap:        с {first_post_gap}")
print(f"   Пауз всего:       {meta_ts['n_pause_months'].iloc[0]}")
print(f"   Дырок в середине: {meta_ts['has_mid_history_gaps'].iloc[0]}")

⚠️ Исключаем неполный период 2026-08 из global
💾 time_series_global.parquet (75 строк, пропусков: 4, после gap: 70)
⚠️ Исключаем неполный период 2026-08-03/2026-08-09 из recent_6q
💾 time_series_recent_6q.parquet (78 строк, пропусков: 2, после gap: 78)
💾 time_series_last_90d.parquet (91 строк, пропусков: 39, после gap: 91)

📊 time_series_meta.parquet:
   Первое событие:   2020-05
   После gap:        с 2020-10
   Пауз всего:       4
   Дырок в середине: False


# 4. TIME: 4 НЕЗАВИСИМЫХ СЛОЯ ДЕТЕКЦИИ + ШТОРМ-АНАЛИЗ


In [5]:
# 4. TIME: 4 НЕЗАВИСИМЫХ СЛОЯ ДЕТЕКЦИИ + ШТОРМ-АНАЛИЗ
# ============================================================================
time_changepoints, time_storms = build_changepoints(df, TIME_PATH)

print(f'\n📊 time_changepoints.parquet: {len(time_changepoints)} строк')
print(time_changepoints.to_string(index=False))

print(f'\n📊 Точки по слоям:')
print(time_changepoints.groupby('layer').size().to_string())

print(f'\n📊 Штормовые эпизоды: {time_storms["episode_id"].dropna().nunique()}')

# --- Просмотр штормовых эпизодов ---
print(f'📊 Штормовые эпизоды:')
episodes = time_storms[time_storms['episode_id'].notna()].groupby('episode_id').agg(
    start=('date', 'min'),
    end=('date', 'max'),
    months=('date', 'count'),
    peak_std=('std_all', 'max'),
).reset_index()
print(episodes.to_string(index=False))



📊 time_changepoints.parquet: 11 строк
   date          method    layer  confidence
2021-02 cosine_identity identity       0.845
2021-08   pelt_behavior behavior       0.175
2022-05 cosine_identity identity       0.825
2023-09      pelt_sound    sound       0.240
2024-02      pelt_sound    sound       0.245
2024-02   pelt_behavior behavior       0.135
2024-02 cosine_identity identity       0.825
2024-07      pelt_sound    sound       0.160
2024-12   pelt_behavior behavior       0.105
2025-10   pelt_behavior behavior       0.150
2026-03   pelt_behavior behavior       0.365

📊 Точки по слоям:
layer
behavior    5
identity    3
sound       3

📊 Штормовые эпизоды: 2
📊 Штормовые эпизоды:
 episode_id      start        end  months  peak_std
          1 2021-01-01 2021-06-01       3  0.144889
          2 2023-10-01 2024-05-01       5  0.207325


# 5. VARIABILITY — ПРОГОН И ЭКСПОРТ

In [6]:
# 5. VARIABILITY — ПРОГОН И ЭКСПОРТ
# ============================================================================
# Считаем Sound Variability через toolkit (VariabilityAnalyzer).
# База: VALID events — скипы не являются вкусом, маскируют сигнал.
# Параметры: окно 12 мес, p_low=75, p_high=99, min_features=2.
# Экспортируем 4 файла: score по месяцам, meta, contributions, эпохи.
# ==============================================================================

# ---------- 1. Готовим valid-базу ----------
# trim_cold_start — отрезает неполный последний месяц и холодный старт
# (gap > 1 мес в первый год). Тот же df_t, что идёт во все детекторы.
df_t = trim_cold_start(df)

# valid_play уже посчитан в Cell 1 (add_valid_play).
# Оставляем только события, где трек дослушан осознанно.
df_valid = df_t[df_t['valid_play']].copy()

# monthly_sound на valid — для калибровки практических порогов.
# Медианы 6 звуковых признаков по месяцам.
monthly_valid = build_monthly_sound(df_valid)

print(f"df_t:            {df_t['period'].nunique()} месяцев")
print(f"df_valid:        {df_valid['period'].nunique()} месяцев, "
      f"{len(df_valid):,} событий ({len(df_valid)/len(df_t)*100:.1f}% от всех)")

# ---------- 2. Прогон пайплайна ----------
# run_variability_pipeline внутри:
#   - считает High/Medium/Low для каждого перехода месяца
#     (Mahalanobis + permutation noise floor + практический гейт)
#   - rolling 12 → score по месяцам
#   - epochs = непрерывные участки rolling > 0.5 длиной >= 2 мес
#     (rolling c min_periods=12 — только полные окна)
#   - feature contributions для High-месяцев
R = run_variability_pipeline(
    df_valid,
    monthly_sound=monthly_valid,
    window=12,
    seed=42,
)

# ---------- 3. Экспорт в parquet ----------
# 4 файла в TIME_PATH:
#   variability.parquet          — month, score
#   variability_meta.parquet     — lifetime/current score, null-границы
#   feature_contributions.parquet — period_to, feature, contribution
#   epochs_sound.parquet         — start, end, duration_months, n_high_months
paths = export_variability(R, TIME_PATH)

# ---------- 4. Что получилось ----------
print(f"\nLifetime score: {R['lifetime_score']:.3f}")
print(f"Current score:  {R['current_score']:.3f}")
print(f"Null границы:   [{R['null_low']:.3f}, {R['null_high']:.3f}]")

# Level по компасу (0.3 / 0.5) — считаем здесь только для контроля.
# В дашборде level берётся из indices_display.json.
def _level(s):
    if s < 0.3: return 'Low'
    if s <= 0.5: return 'Medium'
    return 'High'

print(f"Lifetime level (0.3/0.5): {_level(R['lifetime_score'])}")
print(f"Current level  (0.3/0.5): {_level(R['current_score'])}")

# Эпохи — на полных окнах (min_periods=12)
# print(f"\nEpochs: {len(R['epochs'])}")
for e in R['epochs']:
    print(f"   {e['start']} … {e['end']}  ({e['duration_months']} мес, "
          f"n_high={e.get('n_high_months', '—')})")

print("\nСохранено:")
for k, p in paths.items():
    print(f"   {k:25s}: {p.name}")

df_t:            70 месяцев
df_valid:        70 месяцев, 45,706 событий (57.5% от всех)

Lifetime score: 0.417
Current score:  0.083
Null границы:   [0.333, 0.417]
Lifetime level (0.3/0.5): Medium
Current level  (0.3/0.5): Low
   2021-10 … 2022-06  (9 мес, n_high=—)
   2024-01 … 2024-04  (4 мес, n_high=—)
   2024-06 … 2025-01  (8 мес, n_high=—)

Сохранено:
   variability              : variability.parquet
   meta                     : variability_meta.parquet
   feature_contributions    : feature_contributions.parquet
   epochs_sound             : epochs_sound.parquet


# 6. PORTRAIT — 4 ОСИ КОМПАСА


In [7]:
# 6. PORTRAIT — 4 ОСИ КОМПАСА
# ============================================================================
# Считает 4 оси портрета:
#   Discovery, Retention, Burnout — на valid-базе внутри portrait.py
#   Variability — передаётся снаружи (посчитана в Cell 5 на valid)
#
# Пишет behavior/portrait.parquet (8 строк: 4 оси × 2 stage).
# Level НЕ пишется — считается в дашборде по индексам в indices_display.json.
# ==============================================================================

# Проверяем, что R из Cell 5 есть в памяти
if 'R' not in dir():
    raise RuntimeError(
        "R не найден в памяти. Прогони Cell 5 сначала."
    )

# variability_score — обязательный аргумент.
# Если short_history=True (мало данных) — R['lifetime_score'] = None.
# Portrait запишет None в Variability, дашборд покажет «недоступно».
variability_score = R['lifetime_score']

portrait = build_portrait(
    df,
    windows,
    BEHAVIOR_PATH,
    variability_score=variability_score,
)
axes = build_axes(df, windows)
for stage, vals in axes.items():
    print(f"\n[{stage}]")
    for k, v in vals.items():
        print(f"   {k:16s}: {v}")


print(f"\n📊 portrait.parquet: {len(portrait)} строк")
print(portrait.to_string(index=False))

# Контроль по осям
print(f"\n📊 По стадиям:")
print(portrait.groupby('stage')['axis'].count().to_string())

# Проверка: Variability совпадает со score из Cell 5
var_global = portrait[
    (portrait['axis'] == 'Variability') & (portrait['stage'] == 'global')
]['value'].iloc[0]
print(f"\n📊 Variability global: {var_global}")
print(f"   Совпадает со score из Cell 5: "
      f"{'✅' if abs(var_global - R['lifetime_score']) < 1e-6 else '⚠️'}")


[global]
   Discovery       : 0.6532868742206219
   Retention       : 0.3549564159740523
   Burnout         : 0.42839143217135656
   short_history   : False

[recent_6q]
   Discovery       : 0.4734620024125452
   Retention       : 0.3382578664620107
   Burnout         : 0.42839143217135656
   short_history   : False

📊 portrait.parquet: 8 строк
       axis    value  short_history     stage
  Discovery 0.653287          False    global
  Retention 0.354956          False    global
    Burnout 0.428391          False    global
Variability 0.416667          False    global
  Discovery 0.473462          False recent_6q
  Retention 0.338258          False recent_6q
    Burnout 0.428391          False recent_6q
Variability 0.416667          False recent_6q

📊 По стадиям:
stage
global       4
recent_6q    4

📊 Variability global: 0.4166666666666667
   Совпадает со score из Cell 5: ✅


# 7. BEHAVIOR: ОБСЕССИИ


In [8]:
# 7. BEHAVIOR: ОБСЕССИИ
# ============================================================================
# Strong: серии >=5 повторов подряд → obsessions_strong.parquet
# Base:   доля событий периода в obsession-парах → 3 файла по окнам
# База:   все события (одна формула на весь проект)
# ==============================================================================


# --- 1. Strong ---
strong = build_strong_obsessions(df)
strong.to_parquet(BEHAVIOR_PATH / 'obsessions_strong.parquet', index=False)
print(f'💾 obsessions_strong.parquet ({len(strong)} серий)')

# --- 2. Base — три окна ---
FREQ = {'global': 'M', 'recent_6q': 'W', 'last_90d': 'D'}

for label, (start, end) in windows.items():
    df_w = df[(df['ts_naive'] >= start) & (df['ts_naive'] <= end)].copy()

    if df_w.empty:
        base = pd.DataFrame(columns=['date', 'obsession_share'])
    else:
        df_w['is_base_obsession_event'] = is_obsession_event(df_w)
        df_w['period'] = df_w['ts_naive'].dt.to_period(FREQ[label])
        base = (df_w.groupby('period')['is_base_obsession_event']
                     .mean()
                     .reset_index(name='obsession_share'))
        base['date'] = base['period'].dt.to_timestamp()
        base = base[['date', 'obsession_share']].sort_values('date')

    out = BEHAVIOR_PATH / f'obsessions_base_{label}.parquet'
    base.to_parquet(out, index=False)
    print(f'💾 {out.name} ({len(base)} периодов)')

💾 obsessions_strong.parquet (49 серий)
💾 obsessions_base_global.parquet (72 периодов)
💾 obsessions_base_recent_6q.parquet (77 периодов)
💾 obsessions_base_last_90d.parquet (54 периодов)


# 8. BEHAVIOR: ВОРОНКА ВЗАИМОДЕЙСТВИЯ

In [9]:
# 8. BEHAVIOR: ВОРОНКА ВЗАИМОДЕЙСТВИЯ
# ============================================================================
# Два среза:
#   - по событиям: сколько событий на каждом этапе
#   - по трекам: сколько уникальных треков на каждом этапе
#
# Этапы:
#   All → Valid (≥30сек, не скип, ≥50% прослушано)
#       → Repeat (трек уже слушался ранее)
#       → Saved (трек в my_music)
#
# База: все события df. Окна не применяются — воронка про всю историю.
# first_valid_seen — колонка из enrich (одна логика «первого валидного»).
# Пишет в BEHAVIOR_PATH:
#   behavior_funnel_events.parquet
#   behavior_funnel_tracks.parquet
# ==============================================================================

# ---------- 1. События по этапам ----------
total_events = len(df)
valid_events = df[df['valid_play']]

# Повторные события: валидные, но трек уже слушался раньше.
# first_valid_seen — из enrich, не считаем заново.
repeat_events = df[
    (df['valid_play']) & (df['ts_naive'] > df['first_valid_seen'])
]

# Сохранённые события: валидные + трек в my_music & repeat
saved_events = df[
    (df['valid_play']) & (df['my_music'])
    & (df['ts_naive'] > df['first_valid_seen'])
]
# ---------- 2. Уникальные треки ----------
unique_all    = df['spotify_track_uri'].nunique()
unique_valid  = df[df['valid_play']]['spotify_track_uri'].nunique()
unique_repeat = repeat_events['spotify_track_uri'].nunique()
unique_saved  = saved_events['spotify_track_uri'].nunique()

# ---------- 3. Таблицы ----------
funnel_events = pd.DataFrame([
    {'stage': 'All events',        'count': total_events,
     'share_from_first': 100.0,
     'share_from_previous': None},
    {'stage': 'Valid plays',       'count': len(valid_events),
     'share_from_first': len(valid_events) / total_events * 100,
     'share_from_previous': len(valid_events) / total_events * 100},
    {'stage': 'Repeat plays',      'count': len(repeat_events),
     'share_from_first': len(repeat_events) / total_events * 100,
     'share_from_previous': len(repeat_events) / len(valid_events) * 100},
    {'stage': 'Saved (my music)',  'count': len(saved_events),
     'share_from_first': len(saved_events) / total_events * 100,
     'share_from_previous': len(saved_events) / len(repeat_events) * 100},
])

funnel_tracks = pd.DataFrame([
    {'stage': 'All unique tracks', 'count': unique_all,
     'share_from_first': 100.0,
     'share_from_previous': None},
    {'stage': 'Valid unique',      'count': unique_valid,
     'share_from_first': unique_valid / unique_all * 100,
     'share_from_previous': unique_valid / unique_all * 100},
    {'stage': 'Repeated unique',   'count': unique_repeat,
     'share_from_first': unique_repeat / unique_all * 100,
     'share_from_previous': unique_repeat / unique_valid * 100
                            if unique_valid else 0},
    {'stage': 'Saved unique',      'count': unique_saved,
     'share_from_first': unique_saved / unique_all * 100,
     'share_from_previous': unique_saved / unique_repeat * 100
                            if unique_repeat else 0},
])

# ---------- 4. Сохранение ----------
funnel_events.to_parquet(
    BEHAVIOR_PATH / 'behavior_funnel_events.parquet', index=False
)
funnel_tracks.to_parquet(
    BEHAVIOR_PATH / 'behavior_funnel_tracks.parquet', index=False
)

print('💾 behavior_funnel_events.parquet')
print(funnel_events.to_string(index=False))
print()
print('💾 behavior_funnel_tracks.parquet')
print(funnel_tracks.to_string(index=False))

💾 behavior_funnel_events.parquet
           stage  count  share_from_first  share_from_previous
      All events  79752        100.000000                  NaN
     Valid plays  45841         57.479436            57.479436
    Repeat plays  30452         38.183368            66.429615
Saved (my music)  22737         28.509630            74.665047

💾 behavior_funnel_tracks.parquet
            stage  count  share_from_first  share_from_previous
All unique tracks  25624        100.000000                  NaN
     Valid unique  15389         60.056978            60.056978
  Repeated unique   6710         26.186388            43.602573
     Saved unique   3349         13.069778            49.910581


#  9. BEHAVIOR: КАРТА СКИПОВ (источники + жанры)

In [10]:
# 9. BEHAVIOR: КАРТА СКИПОВ (источники + жанры)
# ============================================================================
# По трём окнам: global / recent_6q / last_90d.
#
# По источникам: My Music / Foreign / Orphan
#   — source уже в df (enrich.add_track_source)
# По жанрам: explode genres_normalized, агрегат по жанру.
#
# Полное прослушивание: ms_played >= duration_ms * 0.9
#
# База: все события df внутри окна.
# Пишет:
#   behavior_skip_by_source.parquet
#   behavior_skip_by_genre.parquet
# ==============================================================================

source_rows = []
genre_rows = []

for stage, (start, end) in windows.items():
    df_w = df[
        (df['ts_naive'] >= start) & (df['ts_naive'] <= end)
    ].copy()

    # Полное прослушивание (>=90% длительности)
    df_w['full_play'] = (
        df_w['ms_played'] >= df_w['duration_ms'] * 0.9
    ).astype(int)

    # ---------- Источники ----------
    # source уже посчитан в enrich.add_track_source
    source_stats = df_w.groupby('source').agg(
        total=('ts', 'count'),
        skip_rate=('is_skip', 'mean'),
        full_play_rate=('full_play', 'mean'),
        valid_rate=('valid_play', 'mean'),
    ).reset_index()
    source_stats['stage'] = stage
    source_rows.append(source_stats)

    # ---------- Жанры ----------
    genre_events = df_w[
        df_w['genres_normalized'].notna()
        & (df_w['genres_normalized'] != 'unknown')
    ].copy()
    genre_events['genre_list'] = genre_events['genres_normalized'].str.split(',')
    genre_events = genre_events.explode('genre_list')
    genre_events['genre'] = genre_events['genre_list'].str.strip()
    genre_events = genre_events[genre_events['genre'] != '']

    genre_stats = genre_events.groupby('genre').agg(
        total=('ts', 'count'),
        skip_rate=('is_skip', 'mean'),
        full_play_rate=('full_play', 'mean'),
    ).reset_index()
    genre_stats['stage'] = stage
    genre_rows.append(genre_stats)

# ---------- Сборка и сохранение ----------
source_combined = pd.concat(source_rows, ignore_index=True)
genre_combined = pd.concat(genre_rows, ignore_index=True)

source_combined.to_parquet(
    BEHAVIOR_PATH / 'behavior_skip_by_source.parquet', index=False
)
genre_combined.to_parquet(
    BEHAVIOR_PATH / 'behavior_skip_by_genre.parquet', index=False
)

print(f'💾 behavior_skip_by_source.parquet ({len(source_combined)} строк, '
      f'{source_combined["stage"].nunique()} окон)')
print(f'💾 behavior_skip_by_genre.parquet ({len(genre_combined)} строк, '
      f'{genre_combined["stage"].nunique()} окон)')

print('\n📊 По источникам:')
print(source_combined.round(3).to_string(index=False))

💾 behavior_skip_by_source.parquet (9 строк, 3 окон)
💾 behavior_skip_by_genre.parquet (1333 строк, 3 окон)

📊 По источникам:
  source  total  skip_rate  full_play_rate  valid_rate     stage
 foreign   4444      0.380           0.559       0.563    global
my_music  43437      0.305           0.622       0.624    global
  orphan  31871      0.410           0.503       0.509    global
 foreign   1591      0.316           0.667       0.668 recent_6q
my_music  17869      0.280           0.702       0.704 recent_6q
  orphan   9279      0.360           0.628       0.627 recent_6q
 foreign    234      0.128           0.838       0.842  last_90d
my_music   1271      0.248           0.725       0.733  last_90d
  orphan    759      0.258           0.722       0.726  last_90d



# 10. BEHAVIOR: КОГОРТНОЕ УДЕРЖАНИЕ НОВИНОК

In [11]:
# 10. BEHAVIOR: КОГОРТНОЕ УДЕРЖАНИЕ НОВИНОК
# ============================================================================
# Горизонты: 30 / 90 / 180 дней.
# Когорты: по годам первого валидного появления трека.
#
# Правая цензура: если окно удержания выходит за последнюю дату —
# результат NaN (недостаточно данных, чтобы судить).
#
# База: valid события df.
# Пишет: behavior_cohorts.parquet
# ==============================================================================

# ---------- 1. Валидные события ----------
valid = df[df['valid_play']].copy()

# first_valid_seen уже в df — используем как first_seen
valid['first_seen'] = valid['first_valid_seen']

# ---------- 2. Когорта = год первого появления ----------
valid['first_year'] = valid['first_seen'].dt.to_period('Y')

first_events = valid[valid['ts_naive'] == valid['first_seen']].copy()
first_events['cohort'] = first_events['first_year'].astype(str)

repeats = valid[valid['ts_naive'] > valid['first_seen']].copy()

# ---------- 3. Функция удержания ----------
def retention_rate_yearly(cohort_df, repeats_df, days):
    """
    Для каждой годовой когорты — доля треков, вернувшихся
    в течение `days` дней после первого появления.

    Правая цензура: если окно выходит за последнюю дату — NaN.
    """
    result = {}
    last_date = valid['ts_naive'].max()

    for cohort, grp in cohort_df.groupby('cohort'):
        total = grp['spotify_track_uri'].nunique()
        if total == 0:
            continue

        grp = grp.copy()
        grp['window_end'] = grp['first_seen'] + pd.Timedelta(days=days)

        if grp['window_end'].max() > last_date:
            result[cohort] = np.nan
            continue

        rep_merged = repeats_df.merge(
            grp[['spotify_track_uri', 'window_end']],
            on='spotify_track_uri', how='inner'
        )

        valid_rep = rep_merged[
            (rep_merged['ts_naive'] > rep_merged['first_seen']) &
            (rep_merged['ts_naive'] <= rep_merged['window_end'])
        ]

        survived = valid_rep['spotify_track_uri'].nunique()
        result[cohort] = survived / total * 100

    return pd.Series(result)


# ---------- 4. Три горизонта ----------
ret30  = retention_rate_yearly(first_events, repeats, 30)
ret90  = retention_rate_yearly(first_events, repeats, 90)
ret180 = retention_rate_yearly(first_events, repeats, 180)

# ---------- 5. Итоговая таблица ----------
cohort_table = pd.DataFrame({
    'cohort':        ret30.index.astype(str),
    'retention_30d': ret30.values,
    'retention_90d': ret90.values,
    'retention_180d': ret180.values,
}).sort_values('cohort').reset_index(drop=True)

cohort_table.to_parquet(
    BEHAVIOR_PATH / 'behavior_cohorts.parquet', index=False
)

print(f'💾 behavior_cohorts.parquet ({len(cohort_table)} строк)')
print(cohort_table.to_string(index=False))

💾 behavior_cohorts.parquet (7 строк)
cohort  retention_30d  retention_90d  retention_180d
  2020      21.413721      27.858628       33.679834
  2021      24.844961      31.860465       33.992248
  2022      34.383954      41.451767       44.078319
  2023      28.906851      34.334103       36.335643
  2024      37.492279      42.310068       44.904262
  2025      30.577889      36.834171       39.974874
  2026            NaN            NaN             NaN


# 11. COLLECTION: АВТОПОРТРЕТЫ ПЛЕЙЛИСТОВ

In [12]:
# 11. COLLECTION: АВТОПОРТРЕТЫ ПЛЕЙЛИСТОВ
# ============================================================================
# Для каждого плейлиста — портрет: медианы аудио, жанры, эпоха,
# вовлечённость (сколько слушалось, skip rate, full play).
#
# Плюс отдельная таблица участников collaborative/contributed.
#
# База: tracks_exportify (метаданные) + valid events (вовлечённость).
# Пишет:
#   playlist_portraits.parquet
#   playlist_collaborators.parquet
# ==============================================================================

# ---------- 1. Агрегаты по трекам из событий ----------
valid_events_df = df[df['valid_play']].copy()

events_agg = df.groupby('spotify_track_uri').agg(
    total_events=('ts', 'count'),
    valid_events=('valid_play', 'sum'),
    skip_events=('is_skip', 'sum'),
).reset_index()

# full_play_share по валидным событиям
valid_full = (valid_events_df
    .assign(full_play=(valid_events_df['ms_played']
                       >= valid_events_df['duration_ms'] * 0.9))
    .groupby('spotify_track_uri')['full_play']
    .mean()
    .reset_index()
)
valid_full.columns = ['spotify_track_uri', 'full_play_share_valid']

events_agg = events_agg.merge(valid_full, on='spotify_track_uri', how='left')
events_agg['skip_rate_all_events'] = (
    events_agg['skip_events'] / events_agg['total_events']
)

# ---------- 2. Уникальные URI по плейлистам ----------
pl_unique = playlist_links.drop_duplicates(
    ['name_norm', 'Track URI']
)[['name_norm', 'Track URI', 'playlist_type']]

# ---------- 3. Портрет каждого плейлиста ----------
rows = []
for name_norm, group in pl_unique.groupby('name_norm'):
    uris = group['Track URI'].dropna().unique()
    total_tracks = len(uris)

    tracks = tracks_exportify[
        tracks_exportify['spotify_track_uri'].isin(uris)
    ]
    n_tracks = len(tracks)
    if n_tracks == 0:
        continue

    ptype = group['playlist_type'].iloc[0]

    # Эпоха
    release_year = pd.to_datetime(tracks['release_date'], errors='coerce').dt.year
    mode_year = (release_year.mode().iloc[0]
                 if not release_year.mode().empty else np.nan)
    decade_2020s    = (release_year >= 2020).mean()
    decade_2010s    = ((release_year >= 2010) & (release_year < 2020)).mean()
    decade_2000s    = ((release_year >= 2000) & (release_year < 2010)).mean()
    decade_pre_2000 = (release_year < 2000).mean()

    # Медианы аудио
    audio_cols = ['valence', 'energy', 'danceability', 'acousticness',
                  'instrumentalness', 'tempo', 'popularity']
    medians = tracks[audio_cols].median().round(3).to_dict()

    # Жанры
    genres = tracks['genres_normalized'].dropna().str.split(',').explode().str.strip()
    genres = genres[genres != 'unknown']
    top_genres = genres.value_counts().head(5).index.tolist()
    unique_genres = genres.nunique()
    genre_diversity = unique_genres / n_tracks if n_tracks > 0 else 0

    # Вовлечённость
    ev = events_agg[events_agg['spotify_track_uri'].isin(uris)]
    total_valid_listens = int(ev['valid_events'].sum())
    unique_listened = int((ev['valid_events'] > 0).sum())
    share_listened = (unique_listened / total_tracks) if total_tracks > 0 else 0
    skip_rate_all = ev['skip_rate_all_events'].mean() if len(ev) > 0 else np.nan
    full_play_share_valid = ev['full_play_share_valid'].mean() if len(ev) > 0 else np.nan

    # Топ-3 трека по валидным прослушиваниям
    track_counts = (
        valid_events_df[valid_events_df['spotify_track_uri'].isin(uris)]
        .groupby(['spotify_track_uri', 'track_name', 'artist_name'])
        .size().nlargest(3).reset_index(name='count')
    )
    top_tracks_str = '; '.join(
        f"{row['track_name']} — {row['artist_name']} ({row['count']})"
        for _, row in track_counts.iterrows()
    )

    rows.append({
        'playlist_name':        name_norm,
        'playlist_type':        ptype,
        'total_tracks':         total_tracks,
        'mode_release_year':    mode_year,
        'decade_2020s':         decade_2020s,
        'decade_2010s':         decade_2010s,
        'decade_2000s':         decade_2000s,
        'decade_pre_2000':      decade_pre_2000,
        'valence_med':          medians['valence'],
        'energy_med':           medians['energy'],
        'danceability_med':     medians['danceability'],
        'acousticness_med':     medians['acousticness'],
        'instrumentalness_med': medians['instrumentalness'],
        'tempo_med':            medians['tempo'],
        'popularity_med':       medians['popularity'],
        'top_genres':           ', '.join(top_genres),
        'unique_genres':        unique_genres,
        'genre_diversity':      genre_diversity,
        'share_listened':       share_listened,
        'skip_rate_all_events': skip_rate_all,
        'full_play_share_valid': full_play_share_valid,
        'total_valid_listens':  total_valid_listens,
        'top_tracks':           top_tracks_str,
    })

portraits_df = pd.DataFrame(rows)
portraits_df.to_parquet(
    COLLECTION_PATH / 'playlist_portraits.parquet', index=False
)
print(f'💾 playlist_portraits.parquet ({len(portraits_df)} плейлистов)')

# ---------- 4. Участники collaborative / contributed ----------
collab_links = playlist_links[
    playlist_links['playlist_type'].isin(['collaborative', 'contributed'])
].copy()
owner_map = (playlists_metadata.drop_duplicates('name_norm')
             .set_index('name_norm')['owner_id'].to_dict())
OWNER_IDS = set(owner_ids['owner_id'].dropna())
owner_name_map = dict(zip(owner_ids['owner_id'], owner_ids['display_name']))

# Единый список внешних участников по всем collab-плейлистам.
# External_N должен быть стабилен между плейлистами.
all_external_ids = sorted(
    set(collab_links['Added By'].dropna().unique())
    - OWNER_IDS
    - {'unknown'}
)
external_id_to_label = {
    ext_id: f'External_{i+1}' for i, ext_id in enumerate(all_external_ids)
}

participant_rows = []
for name_norm, group in collab_links.groupby('name_norm'):
    owner_id = owner_map.get(name_norm)
    added_counts = group['Added By'].value_counts(dropna=False)
    for added_by, count in added_counts.items():
        if pd.isna(added_by) or added_by == 'unknown':
            display_name = 'Unknown'
            role = 'unknown'
        elif added_by in OWNER_IDS:
            display_name = owner_name_map.get(added_by, added_by)
            if owner_id is not None and added_by == owner_id:
                role = 'owner'
            elif owner_id is not None:
                role = 'collaborator'
            else:
                role = 'contributor'
        else:
            display_name = external_id_to_label.get(added_by, 'External_unknown')
            role = 'external'


        participant_rows.append({
            'playlist_name': name_norm,
            'added_by_id':   str(added_by) if not pd.isna(added_by) else 'unknown',
            'display_name':  display_name,
            'additions':     count,
            'role':          role,
        })

participants_df = pd.DataFrame(participant_rows)
participants_df.to_parquet(
    COLLECTION_PATH / 'playlist_collaborators.parquet', index=False
)
print(f'💾 playlist_collaborators.parquet ({len(participants_df)} записей)')

💾 playlist_portraits.parquet (296 плейлистов)
💾 playlist_collaborators.parquet (48 записей)


# 12. COLLECTION: ТИПЫ НАПОЛНЕНИЯ И АКТИВНОСТЬ

In [13]:
# 12. COLLECTION: ТИПЫ НАПОЛНЕНИЯ И АКТИВНОСТЬ
# ============================================================================
# fill_type (4 типа): как плейлист наполнялся.
#   mass_import / batch / curated / empty
#
# activity_group (4 типа): как плейлист живёт сейчас.
#   active_listened / active_not_listened /
#   abandoned_but_listened / abandoned_not_listened
#
# База: valid events + playlist_links + playlists_metadata.
# Пишет:
#   collection_fill_summary.parquet
#   collection_activity_scatter.parquet
# ==============================================================================

# ---------- 1. Fill type для personal / collaborative ----------
own_links = playlist_links[
    playlist_links['playlist_type'].isin(['personal', 'collaborative'])
].copy()
own_links['Added At'] = (pd.to_datetime(own_links['Added At'], errors='coerce')
                            .dt.tz_localize(None))

# first_valid_seen уже в df — используем как карту
first_valid_map = (df[df['valid_play']]
                     .groupby('spotify_track_uri')['first_valid_seen']
                     .first())
own_links['first_valid_seen'] = own_links['Track URI'].map(first_valid_map)
own_links['listened_before'] = own_links['first_valid_seen'] < own_links['Added At']

# Пороги классификации
MASS_IMPORT_MIN_SIZE         = 100
MASS_IMPORT_MAX_LISTENED     = 0.30
MASS_IMPORT_MAX_DURATION_MIN = 15
PACKET_INTERVAL_MIN          = 2


def classify_playlist_for_fill(group):
    total = len(group)
    if total == 0:
        return 'empty'

    group = group.sort_values('Added At').reset_index(drop=True)
    n_days = group['Added At'].dt.date.nunique()

    if n_days == 1:
        day_duration = (group['Added At'].max()
                        - group['Added At'].min()).total_seconds() / 60
        known_mask = group['listened_before'].notna()
        listened_before_share = (group.loc[known_mask, 'listened_before'].mean()
                                 if known_mask.sum() > 0 else 0.0)
        if (total >= MASS_IMPORT_MIN_SIZE
                and listened_before_share < MASS_IMPORT_MAX_LISTENED
                and day_duration <= MASS_IMPORT_MAX_DURATION_MIN):
            return 'mass_import'

    series_lengths = []
    current_len = 1
    for i in range(1, total):
        interval = (group.loc[i, 'Added At']
                    - group.loc[i - 1, 'Added At']).total_seconds() / 60
        if interval <= PACKET_INTERVAL_MIN:
            current_len += 1
        else:
            series_lengths.append(current_len)
            current_len = 1
    series_lengths.append(current_len)

    has_single = any(length == 1 for length in series_lengths)
    return 'curated' if has_single else 'batch'


fill_rows = []
for name, group in own_links.groupby('name_norm'):
    fill_rows.append({
        'name_norm': name,
        'fill_type': classify_playlist_for_fill(group),
    })
fill_df = pd.DataFrame(fill_rows)

# ---------- 2. Активность personal / collaborative ----------
valid_events_act = df[df['valid_play']].copy()
track_listen_counts = (valid_events_act
    .groupby('spotify_track_uri')
    .size()
    .reset_index(name='valid_listens')
)

pl_tracks = own_links[['name_norm', 'playlist_type', 'Track URI']].drop_duplicates()
pl_tracks = pl_tracks.merge(
    track_listen_counts,
    left_on='Track URI', right_on='spotify_track_uri', how='left'
)
pl_tracks['valid_listens'] = pl_tracks['valid_listens'].fillna(0).astype(int)

playlist_agg = pl_tracks.groupby(['name_norm', 'playlist_type']).agg(
    total_listens=('valid_listens', 'sum'),
    unique_tracks=('Track URI', 'nunique'),
).reset_index()

# Метаданные: давность обновления
meta = (playlists_metadata.drop_duplicates('name_norm')
        [['name_norm', 'num_tracks', 'last_modified']].copy())
meta['last_modified_dt'] = pd.to_datetime(meta['last_modified'], errors='coerce')
max_date_meta = meta['last_modified_dt'].max()
meta['days_since_update'] = (
    max_date_meta - meta['last_modified_dt']
).dt.days

pl_activity = playlist_agg.merge(
    meta[['name_norm', 'num_tracks', 'days_since_update']],
    on='name_norm', how='left'
)
pl_activity = pl_activity.merge(fill_df, on='name_norm', how='left')

# Классификация активности
median_listens = pl_activity['total_listens'].median()


def classify_activity(row):
    recent = row['days_since_update'] <= 365
    listened = row['total_listens'] > median_listens
    if recent and listened:
        return 'active_listened'
    elif recent and not listened:
        return 'active_not_listened'
    elif not recent and listened:
        return 'abandoned_but_listened'
    else:
        return 'abandoned_not_listened'


pl_activity['activity_group'] = pl_activity.apply(classify_activity, axis=1)

# ---------- 3. Сохранение ----------
fill_summary = pl_activity.groupby('fill_type').agg(
    playlists=('name_norm', 'nunique'),
    avg_size=('unique_tracks', 'mean'),
    total_listens=('total_listens', 'sum'),
).reset_index()
fill_summary['share'] = (
    fill_summary['playlists'] / fill_summary['playlists'].sum() * 100
)

fill_summary.to_parquet(
    COLLECTION_PATH / 'collection_fill_summary.parquet', index=False
)
pl_activity.to_parquet(
    COLLECTION_PATH / 'collection_activity_scatter.parquet', index=False
)

print(f'💾 collection_fill_summary.parquet ({len(fill_summary)} типов)')
print(fill_summary.round(1).to_string(index=False))
print(f'\n💾 collection_activity_scatter.parquet ({len(pl_activity)} плейлистов)')

💾 collection_fill_summary.parquet (3 типов)
  fill_type  playlists  avg_size  total_listens  share
      batch         55      49.7           3783   35.0
    curated         87      83.0          47614   55.4
mass_import         15     223.3            730    9.6

💾 collection_activity_scatter.parquet (157 плейлистов)


# 13. KPI_SUMMARY: СБОРКА ВСЕХ ИНДЕКСОВ

In [14]:
# 13. KPI_SUMMARY: СБОРКА ВСЕХ ИНДЕКСОВ
# ============================================================================
# Оркестратор build_kpi_summary из src/indices.py.
# Вызывает ~40 индексов × 3 окна → 106 строк.
# База: df обогащён в Cell 1 (valid_play, source, period, session_id).
# Пишет: dashboard/kpi_summary.parquet
# ==============================================================================

kpi = build_kpi_summary(df, windows, tracks_exportify, playlist_links)

out = DASHBOARD_DATA_PATH / 'kpi_summary.parquet'
kpi.to_parquet(out, index=False)

print(f'💾 {out.name} ({len(kpi)} строк)')
print(f'   global:    {(kpi["stage"]=="global").sum()}')
print(f'   recent_6q: {(kpi["stage"]=="recent_6q").sum()}')
print(f'   last_90d:  {(kpi["stage"]=="last_90d").sum()}')
print(f'\nПо категориям:')
print(kpi.groupby('category').size().to_string())

💾 kpi_summary.parquet (106 строк)
   global:    39
   recent_6q: 34
   last_90d:  33

По категориям:
category
basic         40
behavioral    25
playlist      20
temporal      21


# 14. ЭКСПОРТ ОБОГАЩЁННОГО df ДЛЯ ДАШБОРДА

In [15]:
# ЯЧЕЙКА 14. ЭКСПОРТ ОБОГАЩЁННОГО df ДЛЯ ДАШБОРДА
# ============================================================================
# Дашборд (app.py) сейчас сам обогащает eda_ready.parquet — дублирует 5 функций
# из src/enrich.py. Выносим в prep: считаем один раз, app читает готовое.
#
# Сохраняем slim-копию: только колонки, которые использует app.
# eda_ready.parquet остаётся сырым источником — не трогаем.
#
# Конфликт: в eda_ready есть колонка 'source' (exportify / exportify_orphans).
# enrich.add_track_source перезаписывает её на my_music / foreign / orphan.
# Старое значение сохраняем как 'data_origin'.
# ==============================================================================


# Колонки для app
COLS_FOR_DASHBOARD = [
    # Идентификация
    'account',
    'spotify_track_uri', 'track_name', 'artist_name',

    # Время
    'ts', 'ts_naive', 'period',
    'hour', 'weekday', 'year', 'month',  # month из eda_ready (число 1-12)

    # Флаги (enrich)
    'valid_play', 'my_music', 'first_valid_seen',
    'source',  # my_music / foreign / orphan
    'session_id',

    # Событие
    'is_skip', 'ms_played', 'played_frac', 'duration_ms',
    'reason_end', 'platform',

    # Аудио
    'valence', 'energy', 'danceability',
    'acousticness', 'instrumentalness', 'tempo',
    'loudness', 'popularity',

    # Метаданные трека
    'genres_normalized', 'release_date',
]

# Проверяем, что все колонки есть
missing = [c for c in COLS_FOR_DASHBOARD if c not in df.columns]
if missing:
    print(f"⚠️ Отсутствуют колонки: {missing}")
    COLS_FOR_DASHBOARD = [c for c in COLS_FOR_DASHBOARD if c in df.columns]

# Сохраняем
out = DASHBOARD_DATA_PATH / 'eda_enriched.parquet'
df[COLS_FOR_DASHBOARD].to_parquet(out, index=False)

print(f'💾 {out.name} ({len(df):,} строк × {len(COLS_FOR_DASHBOARD)} колонок)')
print(f'   valid_play:      {df["valid_play"].sum():,}')
print(f'   my_music:        {df["my_music"].sum():,}')
print(f'   с session_id:    {df["session_id"].notna().sum():,}')
print(f'   период:          {df["period"].min()} — {df["period"].max()}')

💾 eda_enriched.parquet (79,752 строк × 32 колонок)
   valid_play:      45,841
   my_music:        43,437
   с session_id:    79,752
   период:          2020-05 — 2026-08


# 15. ПРОВЕРКА ПАРКЕТОВ ДАШБОРДА


In [16]:
# 15. ПРОВЕРКА ПАРКЕТОВ ДАШБОРДА
# ============================================================================
# Официальная проверка всех паркетов, которые читает дашборд.
# Проверяет:
#   1. Файл существует.
#   2. Все обязательные колонки на месте.
#   3. Файл не пустой.
#   4. Нет полностью пустых колонок.
#   5. Ключи уникальны (где заданы).
#
# Не падает на отсутствующих файлах — собирает отчёт.
# ==============================================================================

# ---------- Ожидаемые паркеты и их схемы ----------
EXPECTED = {
    # ===== SOUND =====
    'sound/sound_clustering_meta.parquet': {
        'columns': ['k', 'silhouette', 'n_tracks', 'threshold', 'passed'],
        'min_rows': 1,
    },
    'sound/sound_core_global.parquet': {
        'columns': ['track_name', 'artist_name', 'valence', 'energy',
                    'danceability', 'acousticness', 'instrumentalness',
                    'popularity', 'genres_normalized', 'cluster',
                    'play_count', 'pc1', 'pc2', 'distance_to_center',
                    'is_characteristic'],
        'min_rows': 100,
    },
    'sound/sound_core_recent_6q.parquet': {
        'columns': ['track_name', 'artist_name', 'valence', 'energy',
                    'danceability', 'acousticness', 'instrumentalness',
                    'popularity', 'genres_normalized', 'cluster',
                    'play_count', 'pc1', 'pc2', 'distance_to_center',
                    'is_characteristic'],
        'min_rows': 50,
    },
    'sound/sound_core_last_90d.parquet': {
        'columns': ['track_name', 'artist_name', 'valence', 'energy',
                    'danceability', 'acousticness', 'instrumentalness',
                    'popularity', 'genres_normalized', 'cluster',
                    'play_count', 'pc1', 'pc2', 'distance_to_center',
                    'is_characteristic'],
        'min_rows': 10,
    },

    # ===== TIME =====
    'time/time_series_global.parquet': {
        'columns': ['date', 'hours', 'tracks', 'energy_mean', 'valence_mean',
                    'instrumentalness_mean', 'discovery_rate',
                    'completion_rate', 'skip_rate', 'desktop_share',
                    'period', 'is_post_gap'],
        'min_rows': 60,
    },
    'time/time_series_recent_6q.parquet': {
        'columns': ['date', 'hours', 'tracks', 'energy_mean', 'valence_mean',
                    'instrumentalness_mean', 'discovery_rate',
                    'completion_rate', 'skip_rate', 'desktop_share',
                    'period', 'is_post_gap'],
        'min_rows': 50,
    },
    'time/time_series_last_90d.parquet': {
        'columns': ['date', 'hours', 'tracks', 'energy_mean', 'valence_mean',
                    'instrumentalness_mean', 'discovery_rate',
                    'completion_rate', 'skip_rate', 'desktop_share',
                    'period', 'is_post_gap'],
        'min_rows': 60,
    },
    'time/time_series_meta.parquet': {
        'columns': ['first_event_month', 'first_post_gap_month', 'last_month',
                    'pause_months', 'n_pause_months', 'has_mid_history_gaps',
                    'mid_gaps'],
        'min_rows': 1,
    },
    'time/time_changepoints.parquet': {
        'columns': ['date', 'method', 'layer', 'confidence'],
        'min_rows': 5,
    },
    'time/time_storms.parquet': {
        'columns': ['date', 'std_all', 'is_storm', 'episode_id'],
        'min_rows': 60,
    },
    'time/variability.parquet': {
        'columns': ['period', 'score'],
        'min_rows': 60,
    },
    'time/variability_meta.parquet': {
        'columns': ['short_history', 'n_transitions', 'n_valid_rolling_points',
                    'window', 'seed', 'n_boot_transitions', 'n_boot_score',
                    'lifetime_score', 'current_score', 'null_low', 'null_high'],
        'min_rows': 1,
    },
    'time/feature_contributions.parquet': {
        'columns': ['period_to', 'feature', 'contribution'],
        'min_rows': 0,   # допускается пустой при short_history
    },
    'time/epochs_sound.parquet': {
        'columns': ['start', 'end', 'duration_months', 'max_score', 'mean_score'],
        'min_rows': 0,
    },

    # ===== BEHAVIOR =====
    'behavior/portrait.parquet': {
        'columns': ['axis', 'value', 'short_history', 'stage'],
        'min_rows': 8,
        'unique_key': ['axis', 'stage'],
    },
    'behavior/obsessions_strong.parquet': {
        'columns': ['track_uri', 'track_name', 'artist_name',
                    'date', 'repeats', 'hours'],
        'min_rows': 0,
    },
    'behavior/obsessions_base_global.parquet': {
        'columns': ['date', 'obsession_share'],
        'min_rows': 10,
    },
    'behavior/obsessions_base_recent_6q.parquet': {
        'columns': ['date', 'obsession_share'],
        'min_rows': 10,
    },
    'behavior/obsessions_base_last_90d.parquet': {
        'columns': ['date', 'obsession_share'],
        'min_rows': 5,
    },
    'behavior/behavior_funnel_events.parquet': {
        'columns': ['stage', 'count', 'share_from_first', 'share_from_previous'],
        'min_rows': 4,
    },
    'behavior/behavior_funnel_tracks.parquet': {
        'columns': ['stage', 'count', 'share_from_first', 'share_from_previous'],
        'min_rows': 4,
    },
    'behavior/behavior_skip_by_source.parquet': {
        'columns': ['source', 'total', 'skip_rate', 'full_play_rate',
                    'valid_rate', 'stage'],
        'min_rows': 3,
    },
    'behavior/behavior_skip_by_genre.parquet': {
        'columns': ['genre', 'total', 'skip_rate', 'full_play_rate', 'stage'],
        'min_rows': 10,
    },
    'behavior/behavior_cohorts.parquet': {
        'columns': ['cohort', 'retention_30d', 'retention_90d', 'retention_180d'],
        'min_rows': 3,
    },

    # ===== COLLECTION =====
    'collection/playlist_portraits.parquet': {
        'columns': ['playlist_name', 'playlist_type', 'total_tracks',
                    'mode_release_year', 'decade_2020s', 'decade_2010s',
                    'decade_2000s', 'decade_pre_2000', 'valence_med',
                    'energy_med', 'danceability_med', 'acousticness_med',
                    'instrumentalness_med', 'tempo_med', 'popularity_med',
                    'top_genres', 'unique_genres', 'genre_diversity',
                    'share_listened', 'skip_rate_all_events',
                    'full_play_share_valid', 'total_valid_listens',
                    'top_tracks'],
        'min_rows': 10,
    },
    'collection/playlist_collaborators.parquet': {
        'columns': ['playlist_name', 'added_by_id', 'display_name',
                    'additions', 'role'],
        'min_rows': 0,
    },
    'collection/collection_fill_summary.parquet': {
        'columns': ['fill_type', 'playlists', 'avg_size', 'total_listens',
                    'share'],
        'min_rows': 1,
    },
    'collection/collection_activity_scatter.parquet': {
        'columns': ['name_norm', 'playlist_type', 'total_listens',
                    'unique_tracks', 'num_tracks', 'days_since_update',
                    'fill_type', 'activity_group'],
        'min_rows': 10,
    },

    # ===== KPI SUMMARY =====
    'kpi_summary.parquet': {
        'columns': ['id', 'index_name', 'value', 'unit', 'stage',
                    'category', 'audience'],
        'min_rows': 100,
        'unique_key': 'id',
    },

    # ===== ДЛЯ APP =====
    'eda_enriched.parquet': {
        'columns': ['account', 'ts_naive', 'period',
                    'valid_play', 'my_music', 'first_valid_seen',
                    'source', 'session_id',
                    'is_skip', 'ms_played', 'played_frac', 'duration_ms'],
        'min_rows': 1000,
    },
}


# ---------- Прогон проверки ----------
print("=" * 80)
print("ПРОВЕРКА ПАРКЕТОВ ДАШБОРДА")
print("=" * 80)

ok_count = 0
warn_count = 0
err_count = 0

# Группировка по папкам для читаемости
by_folder = {}
for rel_path in EXPECTED:
    folder = rel_path.split('/')[0] if '/' in rel_path else '(корень)'
    by_folder.setdefault(folder, []).append(rel_path)

for folder in ['sound', 'time', 'behavior', 'collection', '(корень)']:
    if folder not in by_folder:
        continue

    print(f"\n📁 {folder}/")
    print("-" * 80)

    for rel_path in sorted(by_folder[folder]):
        spec = EXPECTED[rel_path]
        full_path = DASHBOARD_DATA_PATH / rel_path
        name = rel_path.split('/')[-1]

        if not full_path.exists():
            print(f"  ❌ {name:50s} ФАЙЛ ОТСУТСТВУЕТ")
            err_count += 1
            continue

        try:
            df_p = pd.read_parquet(full_path)
        except Exception as e:
            print(f"  ❌ {name:50s} ОШИБКА ЧТЕНИЯ: {e}")
            err_count += 1
            continue

        n_rows = len(df_p)
        n_cols = len(df_p.columns)

        issues = []

        # Строки
        if n_rows < spec.get('min_rows', 1):
            issues.append(f"строк {n_rows} < {spec['min_rows']}")

        # Колонки
        missing = [c for c in spec['columns'] if c not in df_p.columns]
        if missing:
            issues.append(f"нет колонок: {missing}")

        # Полностью пустые колонки (только среди обязательных)
        empty_cols = [
            c for c in spec['columns']
            if c in df_p.columns and df_p[c].isna().all()
        ]
        if empty_cols:
            issues.append(f"пустые: {empty_cols}")

        # Уникальность ключа
        key = spec.get('unique_key')
        if key:
            key_cols = [key] if isinstance(key, str) else key
            if all(k in df_p.columns for k in key_cols):
                n_dup = df_p.duplicated(subset=key_cols).sum()
                if n_dup > 0:
                    issues.append(f"дубликатов по {key_cols}: {n_dup}")

        if not issues:
            print(f"  ✅ {name:50s} {n_rows:>6} × {n_cols:>2}")
            ok_count += 1
        else:
            print(f"  ⚠️ {name:50s} {n_rows:>6} × {n_cols:>2}")
            for issue in issues:
                print(f"       └─ {issue}")
            warn_count += 1


# ---------- Сводка ----------
print("\n" + "=" * 80)
print("СВОДКА")
print("=" * 80)
print(f"   ✅ OK:      {ok_count}")
print(f"   ⚠️  Проблем: {warn_count}")
print(f"   ❌ Ошибок:  {err_count}")
print(f"   Всего проверено: {ok_count + warn_count + err_count} из {len(EXPECTED)}")

if warn_count == 0 and err_count == 0:
    print("\n🎉 Все паркеты дашборда в порядке.")
else:
    print("\n⚠️ Есть проблемы — см. выше.")

ПРОВЕРКА ПАРКЕТОВ ДАШБОРДА

📁 sound/
--------------------------------------------------------------------------------
  ✅ sound_clustering_meta.parquet                           1 ×  5
  ✅ sound_core_global.parquet                            4374 × 15
  ✅ sound_core_last_90d.parquet                           484 × 15
  ✅ sound_core_recent_6q.parquet                         2486 × 15

📁 time/
--------------------------------------------------------------------------------
  ✅ epochs_sound.parquet                                    3 ×  5
  ✅ feature_contributions.parquet                         162 ×  3
  ✅ time_changepoints.parquet                              11 ×  4
  ✅ time_series_global.parquet                             75 × 12
  ✅ time_series_last_90d.parquet                           91 × 12
  ✅ time_series_meta.parquet                                1 ×  7
  ✅ time_series_recent_6q.parquet                          78 × 12
  ✅ time_storms.parquet                                

In [17]:
# ЯЧЕЙКА 16. ЭКСПОРТ В DATASETS (в git)
# ============================================================================
# 1. Копирует все паркеты из приватной папки в datasets/.
# 2. Маскирует три паркета с личными данными и перезаписывает их.
# ============================================================================

import shutil

# ---------- 1. Копируем всё ----------
for src in DASHBOARD_DATA_PATH.rglob('*.parquet'):
    rel = src.relative_to(DASHBOARD_DATA_PATH)
    dst = DATASETS_PATH / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print(f'📁 Скопировано в datasets/')

# ---------- 2. Маскируем три паркета ----------
masked = mask_all(
    playlist_portraits=portraits_df,
    playlist_collaborators=participants_df,
    collection_activity_scatter=pl_activity,
    # служебные — только для построения маппингов
    playlists_metadata=playlists_metadata,
    owner_ids=owner_ids,
    playlist_links=playlist_links,
)

masked['playlist_portraits'].to_parquet(
    DATASETS_PATH / 'collection' / 'playlist_portraits.parquet', index=False
)
masked['playlist_collaborators'].to_parquet(
    DATASETS_PATH / 'collection' / 'playlist_collaborators.parquet', index=False
)
masked['collection_activity_scatter'].to_parquet(
    DATASETS_PATH / 'collection' / 'collection_activity_scatter.parquet', index=False
)

print('🔒 Замаскированы: playlist_portraits, playlist_collaborators, collection_activity_scatter')

📁 Скопировано в datasets/
🔒 Замаскированы: playlist_portraits, playlist_collaborators, collection_activity_scatter


# Все 29 паркетов: поля и назначение

## 📁 `sound/` — ядро вкуса

### `sound_core_global.parquet` (4374 × 15)
| Поле | Назначение |
|---|---|
| `track_name`, `artist_name` | идентификация |
| `valence`, `energy`, `danceability`, `acousticness`, `instrumentalness` | 5 аудио-признаков (кластеризация) |
| `popularity` | 0–100, для Hipster |
| `genres_normalized` | жанры через запятую |
| `cluster` | номер кластера (обучен на global) |
| `play_count` | валидных прослушиваний в окне |
| `pc1`, `pc2` | PCA-координаты для визуализации |
| `distance_to_center` | расстояние до центра кластера |
| `is_characteristic` | топ-5 ближайших к центру в кластере |

**Назначение:** scatter plot в Sound Personal + таблица кластеров в Professional.

### `sound_core_recent_6q.parquet` (2486 × 15), `sound_core_last_90d.parquet` (484 × 15)
Та же схема. **Назначение:** сравнение состава ядра между окнами.

### `sound_clustering_meta.parquet` (1 × 5)
`k`, `silhouette`, `n_tracks`, `threshold`, `passed`.
**Назначение:** подпись качества кластеризации в Sound Professional.

---

## 📁 `time/` — время и эволюция

### `time_series_global.parquet` (75 × 12)
| Поле | Назначение |
|---|---|
| `period` | Period месяца (`2020-05`) |
| `date` | timestamp начала периода (для plotly) |
| `hours`, `tracks` | объём |
| `energy_mean`, `valence_mean`, `instrumentalness_mean` | звуковые средние |
| `discovery_rate` | доля новых треков |
| `completion_rate` | доля дослушанных ≥90% |
| `skip_rate` | доля скипов (по всем событиям) |
| `desktop_share` | доля десктопа |
| `is_post_gap` | False в холодном старте (серая зона) |

**Назначение:** основной график Time Personal/Professional. **75 строк** — с NaN-строками для дырок (2020-06…09).

### `time_series_recent_6q.parquet` (78 × 12), `time_series_last_90d.parquet` (91 × 12)
Та же схема. Частота: недели / дни.
**Назначение:** переключение периодов в Time.

### `time_series_meta.parquet` (1 × 7)
`first_event_month`, `first_post_gap_month`, `last_month`, `pause_months`, `n_pause_months`, `has_mid_history_gaps`, `mid_gaps`.
**Назначение:** серая зона + подпись «детекторы работают отсюда» в Time.

### `time_changepoints.parquet` (11 × 4)
| Поле | Значение |
|---|---|
| `date` | YYYY-MM |
| `method` | `pelt_sound` / `pelt_behavior` / `cosine_identity` |
| `layer` | `sound` / `behavior` / `identity` |
| `confidence` | bootstrap (0–1) |

**Назначение:** три слоя точек разладки разными цветами в Time Personal.

### `time_storms.parquet` (70 × 4)
`date`, `std_all` (rolling std), `is_storm` (bool), `episode_id` (int или None).
**Назначение:** фоновая линия + заливка штормовых эпизодов.

### `variability.parquet` (69 × 2)
`period`, `score` (0–1).
**Назначение:** график Variability в Time Professional, line chart.

### `variability_meta.parquet` (1 × 11)
`short_history`, `n_transitions`, `n_valid_rolling_points`, `window`, `seed`, `n_boot_transitions`, `n_boot_score`, `lifetime_score`, `current_score`, `null_low`, `null_high`.
**Назначение:** сводка в панель Variability + флаг short_history для дашборда.

### `feature_contributions.parquet` (162 × 3)
`period_to`, `feature`, `contribution`.
**Назначение:** «какие признаки сдвинулись сильнее всего» для топ-5 High-месяцев.

### `epochs_sound.parquet` (3 × 5)
`start`, `end`, `duration_months`, `max_score`, `mean_score`.
**Назначение:** заливка эпох высокой Variability на Time Personal.

---

## 📁 `behavior/` — компас и поведение

### `portrait.parquet` (8 × 4)
`axis` (Discovery / Retention / Burnout / Variability), `value`, `short_history`, `stage` (global / recent_6q).
**Назначение:** компас (radar) в Behavior Personal + карточки осей в Professional.

### `obsessions_strong.parquet` (49 × 6)
`track_uri`, `track_name`, `artist_name`, `date`, `repeats`, `hours`.
**Назначение:** «сильные обсессии» — серии ≥5 подряд. Diamonds на графике в Behavior Personal.

### `obsessions_base_global.parquet` (72 × 2), `obsessions_base_recent_6q.parquet` (77 × 2), `obsessions_base_last_90d.parquet` (54 × 2)
`date`, `obsession_share`.
**Назначение:** bar chart доли obsession-событий по месяцам/неделям/дням.

### `behavior_funnel_events.parquet` (4 × 4), `behavior_funnel_tracks.parquet` (4 × 4)
`stage`, `count`, `share_from_first`, `share_from_previous`.
**Назначение:** воронка взаимодействия (две — по событиям и по трекам) в Behavior Professional.

### `behavior_skip_by_source.parquet` (9 × 6)
`source` (my_music / foreign / orphan), `total`, `skip_rate`, `full_play_rate`, `valid_rate`, `stage`.
**Назначение:** bar chart по источникам.

### `behavior_skip_by_genre.parquet` (1333 × 5)
`genre`, `total`, `skip_rate`, `full_play_rate`, `stage`.
**Назначение:** horizontal bar по жанрам (топ-10).

### `behavior_cohorts.parquet` (7 × 4)
`cohort` (год), `retention_30d`, `retention_90d`, `retention_180d`.
**Назначение:** heatmap удержания в Behavior Professional.

---

## 📁 `collection/` — плейлисты

### `playlist_portraits.parquet` (296 × 23)
`playlist_name`, `playlist_type`, `total_tracks`, `mode_release_year`, `decade_2020s / 2010s / 2000s / pre_2000`, `valence_med`, `energy_med`, `danceability_med`, `acousticness_med`, `instrumentalness_med`, `tempo_med`, `popularity_med`, `top_genres`, `unique_genres`, `genre_diversity`, `share_listened`, `skip_rate_all_events`, `full_play_share_valid`, `total_valid_listens`, `top_tracks`.
**Назначение:** автопортрет выбранного плейлиста в Collection Personal (radar + метаданные).

### `playlist_collaborators.parquet` (48 × 5)
`playlist_name`, `added_by_id`, `display_name`, `additions`, `role` (owner / collaborator / contributor / external / unknown).
**Назначение:** список участников коллаборативных плейлистов.

### `collection_fill_summary.parquet` (3 × 5)
`fill_type` (mass_import / batch / curated / empty), `playlists`, `avg_size`, `total_listens`, `share`.
**Назначение:** таблица типов наполнения в Collection Professional.

### `collection_activity_scatter.parquet` (157 × 8)
`name_norm`, `playlist_type`, `total_listens`, `unique_tracks`, `num_tracks`, `days_since_update`, `fill_type`, `activity_group` (active_listened / active_not_listened / abandoned_but_listened / abandoned_not_listened).
**Назначение:** scatter «жизнь плейлистов» — давность vs прослушивания.

---

## 📁 корень

### `kpi_summary.parquet` (106 × 7)
| Поле | Назначение |
|---|---|
| `id` | уникальный идентификатор (`hipster_index_G`) |
| `index_name` | человекочитаемое (`Hipster Index`) |
| `value` | число |
| `unit` | `index / percent / hours / count / days` |
| `stage` | `global / recent_6q / last_90d` |
| `category` | `basic / temporal / behavioral / playlist` |
| `audience` | `both / personal / professional` |

**Назначение:** ВСЕ карточки индексов во всех вкладках. `indices_display.json` даёт пороги и тексты, `id` связывает.

---

## Как это всё связано

```
kpi_summary.parquet ─── все карточки индексов (106 значений)

portrait.parquet ────── компас (4 оси × 2 stage)

time_changepoints ───── 3 слоя пунктиров на Time
time_storms ─────────── заливка штормов
time_series_meta ────── серая зона холодного старта
time_series_* ───────── линии на графике Time
variability_* ───────── панель Variability
epochs_sound ────────── заливка эпох
feature_contributions ─ топ-признаки High-месяцев

sound_core_* ────────── Sound: scatter + кластеры
sound_clustering_meta ─ качество кластеризации

obsessions_* ────────── Behavior: обсессии
behavior_funnel_* ───── Behavior: воронки
behavior_skip_* ─────── Behavior: скипы
behavior_cohorts ────── Behavior: когорты

playlist_portraits ──── Collection: автопортрет
playlist_collaborators ─ Collection: участники
collection_fill_* ───── Collection: типы наполнения
collection_activity_* ─ Collection: жизнь плейлистов
```

**Всё. Это то, что читает `app.py` для отрисовки.**

# `eda_enriched.parquet` — поля и назначение

## Идентификация

| Поле | Тип | Назначение |
|---|---|---|
| `account` | str | аккаунт (для группировки по сессиям) |
| `spotify_track_uri` | str | уникальный ID трека |
| `track_name` | str | название трека |
| `artist_name` | str | исполнитель |

## Время

| Поле | Тип | Назначение |
|---|---|---|
| `ts` | datetime с TZ | исходный timestamp (для совместимости) |
| `ts_naive` | datetime без TZ | основной timestamp для расчётов |
| `period` | Period('M') | месяц (`2020-05`) — для группировок и графиков |
| `hour` | int 0–23 | час дня — для Night Listening Share |
| `weekday` | int/str | день недели — для анализа расписания |
| `year` | int | год |
| `month` | int 1–12 | **номер месяца года** из eda_ready (число, не Period!) |

**Важно:** `month` и `period` — разные вещи. `month=5` и `period=Period('2020-05')`. Не путать.

## Флаги (созданы в `enrich.py`)

| Поле | Тип | Откуда | Назначение |
|---|---|---|---|
| `valid_play` | bool | `add_valid_play` | осознанное прослушивание (не скип, ≥30 сек, ≥50%) |
| `my_music` | bool | `add_my_music` | трек в лайках или своих плейлистах |
| `first_valid_seen` | datetime без TZ | `add_first_valid_seen` | дата первого валидного прослушивания трека |
| `source` | str | `add_track_source` | **my_music / foreign / orphan** (категория трека) |
| `session_id` | int | `add_sessions` | номер сессии (разрыв >30 мин = новая) |

**Конфликт:** в `eda_ready.parquet` тоже была колонка `source`, но со значениями `exportify / exportify_orphans` (происхождение данных). Она **перезаписана** в `enrich.add_track_source`. Если app её использовал — сломается. Проверить grep.

## Событие

| Поле | Тип | Назначение |
|---|---|---|
| `is_skip` | bool | был ли скип |
| `ms_played` | int | длительность прослушивания в ms |
| `played_frac` | float 0–1 | доля трека, которую дослушали |
| `duration_ms` | int | полная длительность трека |
| `reason_end` | str | причина окончания (trackdone / fwdbtn / …) |
| `platform` | str | платформа (Windows / Android / web_player) |

## Аудио-признаки

| Поле | Тип | Диапазон |
|---|---|---|
| `valence` | float | 0–1 (настроение) |
| `energy` | float | 0–1 (интенсивность) |
| `danceability` | float | 0–1 (танцевальность) |
| `acousticness` | float | 0–1 (акустика) |
| `instrumentalness` | float | 0–1 (без вокала) |
| `tempo` | float | BPM |
| `loudness` | float | dB (обычно −60…0) |
| `popularity` | int | 0–100 |

**Используются:** во всех Sound-индексах, в `time_series` (valence_mean, energy_mean, instrumentalness_mean), в PELT-A, в Variability, в кластеризации.

## Метаданные трека

| Поле | Тип | Назначение |
|---|---|---|
| `genres_normalized` | str | жанры через запятую (`rock, indie, alternative`) |
| `release_date` | str | дата релиза — для Music Recency Index |

**Используются:** жанры в Genre Diversity, skip_by_genre, автопортретах плейлистов. Release date — для Music Recency.

---

## Что НЕ в этом файле (и почему)

**Не сохраняем:**
- `Added At`, `Added By`, `Record Label` — про плейлисты, не про события.
- `ip_addr`, `conn_country`, `offline`, `incognito_mode` — приватность + не используются.
- `key`, `mode`, `liveness`, `speechiness`, `explicit`, `shuffle`, `skipped` — не используются в текущих индексах.
- `total_played_ms`, `max_played_ms` — производные от `ms_played`.
- `listen_count`, `first_listen`, `last_listen` — производные от истории.
- `has_audio_features` — метаданные, не нужны.

**Если понадобятся** — добавить в `COLS_FOR_DASHBOARD` в Cell 14 и пересобрать.

---

## Что читает app из этого файла

| Блок в app | Какие поля нужны |
|---|---|
| **KPI карточки** | не отсюда — из `kpi_summary.parquet` |
| **Time Personal / Professional** | `ts_naive`, `period`, `is_skip`, `ms_played`, `played_frac`, `duration_ms`, `platform`, `valid_play` |
| **Behavior skip-by-source** | не отсюда — из `behavior_skip_by_source.parquet` |
| **Behavior funnels** | не отсюда — из `behavior_funnel_*.parquet` |
| **Overview** | `my_music`, `valid_play`, `source`, `session_id` |
| **Heatmap по часам** (если есть) | `hour`, `weekday` |
| **Sound** | не отсюда — из `sound_core_*.parquet` |
| **Collection** | не отсюда — из `collection/*.parquet` |

**По факту:** `eda_enriched.parquet` заменяет `load_basic_info()` в app — те 5 функций `enrich`, которые сейчас дублируются. app просто читает готовое и не тратит время на обработку 80 000 событий при каждом старте.

---

## Схема в цифрах

- **79 752 строки**
- **30 колонок** (в моём списке `COLS_FOR_DASHBOARD`)
- **~5–8 MB** в parquet (сжатие)

**Проверить:**
```python
import pandas as pd
from pathlib import Path
p = Path(r'D:\Project_2_data_spotify\Code\spotify-listener-portrait\data\stage4_output\dashboard\eda_enriched.parquet')
df = pd.read_parquet(p)
print(f"Размер: {p.stat().st_size / 1024 / 1024:.1f} MB")
print(f"Строк: {len(df):,}, колонок: {len(df.columns)}")
print(f"\nТипы колонок:")
print(df.dtypes.to_string())
```

**Скинь вывод — увидим, что сохранилось.**